# 🛡️ ToxGuard — Multilingual Toxic Comment Classifier

**NeuroLogic '26 | Challenge 3: Multilingual Toxic Comment Classification**

- **Model:** XLM-RoBERTa Base
- **Task:** Binary Toxicity Classification (multilingual)
- **Evaluation Metric:** ROC-AUC
- **Best ROC-AUC:** `0.9918` (Epoch 3)
- **Accuracy:** `95.26%`
- **XAI:** LIME token explanations + Attention heatmaps

---

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install required libraries
!pip install transformers datasets scikit-learn openpyxl accelerate gradio lime matplotlib seaborn -q

## ⚙️ Cell 2 — Imports & Hardware Check

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    confusion_matrix, classification_report, roc_curve
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import Dataset

# Plotting style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

# Hardware Check
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Hardware Check — Using device: {DEVICE}')

## 📂 Cell 3 — Load Dataset (Auto-detect file inside Kaggle folder)

In [ ]:
# ── Kaggle dataset folder paths ──────────────────────────────────────────────
TRAIN_DIR = '/kaggle/input/datasets/ayushtiwari5410/toxic-labeled'
TEST_DIR  = '/kaggle/input/datasets/ayushtiwari5410/toxic-no-label-evaluation'
TEXT_COL  = 'text'
LABEL_COL = 'label'   # ✅ actual column name in this dataset

# ── Auto-detect the CSV / Excel file inside each folder ──────────────────────
def find_file(folder):
    for ext in ['*.csv', '*.xlsx', '*.xls']:
        matches = glob.glob(os.path.join(folder, ext))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'No CSV/Excel file found in: {folder}')

TRAIN_PATH = find_file(TRAIN_DIR)
TEST_PATH  = find_file(TEST_DIR)
print(f'✅ Train file : {TRAIN_PATH}')
print(f'✅ Test  file : {TEST_PATH}')

# ── Load based on extension ───────────────────────────────────────────────────
def load_file(path):
    if path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path)

print('\nLoading dataset files...')
train_df = load_file(TRAIN_PATH)
test_df  = load_file(TEST_PATH)

print(f'Train columns : {train_df.columns.tolist()}')
print(f'Test  columns : {test_df.columns.tolist()}')

# ── Preprocess train ──────────────────────────────────────────────────────────
train_df = train_df[[TEXT_COL, LABEL_COL]].dropna()
train_df[TEXT_COL]  = train_df[TEXT_COL].astype(str).str.strip()
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)

# ── Preprocess test (text only) ───────────────────────────────────────────────
test_df = test_df[[TEXT_COL]].dropna()
test_df[TEXT_COL] = test_df[TEXT_COL].astype(str).str.strip()

print(f'\nTrain samples : {len(train_df)}')
print(f'Test  samples : {len(test_df)}')
print(f'Label distribution:\n{train_df[LABEL_COL].value_counts()}')
train_df.head()

## 📊 Cell 3b — Dataset Distribution Diagrams

In [ ]:
label_counts = train_df[LABEL_COL].value_counts().sort_index()
label_names  = {0: 'Non-Toxic', 1: 'Toxic'}
colors       = ['#22c55e', '#ef4444']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('ToxGuard — Dataset Overview', fontsize=14, fontweight='bold')

# ── 1. Bar chart ─────────────────────────────────────────────────────────────
axes[0].bar([label_names[k] for k in label_counts.index],
            label_counts.values, color=colors, width=0.5, edgecolor='white')
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Sample Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# ── 2. Pie chart ─────────────────────────────────────────────────────────────
axes[1].pie(label_counts.values,
            labels=[label_names[k] for k in label_counts.index],
            colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Balance')

# ── 3. Text-length histogram ──────────────────────────────────────────────────
for label_val, color in zip([0, 1], colors):
    lengths = train_df[train_df[LABEL_COL] == label_val][TEXT_COL].str.split().str.len()
    axes[2].hist(lengths, bins=40, alpha=0.6, color=color,
                 label=label_names[label_val], edgecolor='none')
axes[2].set_title('Text Length Distribution (words)')
axes[2].set_xlabel('Word Count')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig('dataset_overview.png', bbox_inches='tight')
plt.show()
print('✅ Saved dataset_overview.png')

## ✂️ Cell 4 — Train / Validation Split

In [ ]:
RANDOM_SEED = 42
VAL_SIZE    = 0.15   # 85% train / 15% validation

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df[TEXT_COL].tolist(),
    train_df[LABEL_COL].tolist(),
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_df[LABEL_COL].tolist()
)

print(f'Training samples   : {len(train_texts)}')
print(f'Validation samples : {len(val_texts)}')

# Split summary diagram
fig, ax = plt.subplots(figsize=(7, 2))
total = len(train_df)
ax.barh('Split', len(train_texts) / total * 100, color='#3b82f6', label=f'Train ({len(train_texts)})')
ax.barh('Split', len(val_texts) / total * 100,
        left=len(train_texts) / total * 100,
        color='#f59e0b', label=f'Val ({len(val_texts)})')
ax.set_xlim(0, 100)
ax.set_xlabel('% of data')
ax.set_title('Train / Validation Split')
ax.legend(loc='lower right')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.1f}%', ha='center', va='center',
            color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 Cell 5 — Load XLM-RoBERTa Tokenizer & Model

In [ ]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN    = 128

print('Downloading XLM-RoBERTa Brain (This might take a minute)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
print(f'Model loaded: {MODEL_NAME}')

## 🔢 Cell 6 — Tokenise & Build PyTorch Datasets

In [ ]:
print('Translating words into numbers for the AI...')

class ToxicDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ToxicDataset(train_texts, train_labels)
val_dataset   = ToxicDataset(val_texts,   val_labels)
test_dataset  = ToxicDataset(test_df[TEXT_COL].tolist())

print(f'Train dataset size : {len(train_dataset)}')
print(f'Val   dataset size : {len(val_dataset)}')
print(f'Test  dataset size : {len(test_dataset)}')

## 📊 Cell 7 — Custom Metrics (ROC-AUC + Accuracy)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = np.argmax(logits, axis=-1)
    roc   = roc_auc_score(labels, probs)
    acc   = accuracy_score(labels, preds)
    return {'roc_auc': roc, 'accuracy': acc}

## 🚀 Cell 8 — Training Arguments & Trainer

In [ ]:
OUTPUT_DIR = './toxguard_model'

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    eval_strategy               = 'epoch',   # ✅ fixed (was evaluation_strategy)
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'roc_auc',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print('Starting the training process... Watch for the ROC-AUC score!')
trainer.train()

## ✅ Cell 9 — Final Evaluation on Validation Set

In [ ]:
metrics = trainer.evaluate()
print('\n===== FINAL EVALUATION METRICS =====')
print(f"  eval_loss     : {metrics['eval_loss']:.4f}")
print(f"  eval_roc_auc  : {metrics['eval_roc_auc']:.4f}")
print(f"  eval_accuracy : {metrics['eval_accuracy']:.4f}")
print('======================================')

## 📈 Cell 9b — Training Curves + ROC + Confusion Matrix

In [ ]:
# ── Pull training history from trainer logs ───────────────────────────────────
history = trainer.state.log_history
train_loss_log = [(e['epoch'], e['loss'])        for e in history if 'loss' in e and 'eval_loss' not in e]
eval_log       = [(e['epoch'], e['eval_loss'],
                   e['eval_roc_auc'], e['eval_accuracy'])
                  for e in history if 'eval_loss' in e]

epochs_e   = [r[0] for r in eval_log]
eval_loss  = [r[1] for r in eval_log]
roc_scores = [r[2] for r in eval_log]
acc_scores = [r[3] for r in eval_log]

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('ToxGuard — Training Results', fontsize=14, fontweight='bold')

# ── Loss curve ───────────────────────────────────────────────────────────────
if train_loss_log:
    te, tl = zip(*train_loss_log)
    axes[0].plot(te, tl, color='#3b82f6', linewidth=2, label='Train Loss')
axes[0].plot(epochs_e, eval_loss, color='#ef4444', linewidth=2,
             marker='o', markersize=6, label='Val Loss')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# ── ROC-AUC curve ────────────────────────────────────────────────────────────
axes[1].plot(epochs_e, roc_scores, color='#8b5cf6', linewidth=2,
             marker='s', markersize=7, label='Val ROC-AUC')
axes[1].plot(epochs_e, acc_scores, color='#22c55e', linewidth=2,
             marker='^', markersize=7, label='Val Accuracy')
axes[1].set_ylim(0.85, 1.01)
axes[1].set_title('ROC-AUC & Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].legend()
for ep, r, a in zip(epochs_e, roc_scores, acc_scores):
    axes[1].annotate(f'{r:.4f}', (ep, r), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8, color='#8b5cf6')

# ── Confusion matrix on validation set ───────────────────────────────────────
val_preds_out = trainer.predict(val_dataset)
val_probs     = torch.softmax(torch.tensor(val_preds_out.predictions), dim=-1).numpy()[:, 1]
val_preds_cls = np.argmax(val_preds_out.predictions, axis=-1)
cm = confusion_matrix(val_labels, val_preds_cls)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Toxic', 'Toxic'],
            yticklabels=['Non-Toxic', 'Toxic'],
            ax=axes[2], linewidths=0.5)
axes[2].set_title('Confusion Matrix (Validation)')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('training_results.png', bbox_inches='tight')
plt.show()

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(val_labels, val_probs)
auc_val     = roc_auc_score(val_labels, val_probs)
fig2, ax2 = plt.subplots(figsize=(6, 5))
ax2.plot(fpr, tpr, color='#8b5cf6', linewidth=2, label=f'ROC-AUC = {auc_val:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.1, color='#8b5cf6')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve — ToxGuard')
ax2.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', bbox_inches='tight')
plt.show()

# ── Classification report ─────────────────────────────────────────────────────
print('\n', classification_report(val_labels, val_preds_cls,
                                   target_names=['Non-Toxic', 'Toxic']))
print('✅ Saved training_results.png and roc_curve.png')

## 📤 Cell 10 — Generate Submission CSV

In [ ]:
print('Generating predictions on test set...')

raw_preds = trainer.predict(test_dataset)
probs = torch.softmax(torch.tensor(raw_preds.predictions), dim=-1).numpy()[:, 1]

submission = pd.DataFrame({
    'text'             : test_df[TEXT_COL].tolist(),
    'toxic_probability': probs
})

submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv with {len(submission)} rows')

# ── Probability distribution of predictions ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(probs[probs < 0.5], bins=30, color='#22c55e', alpha=0.7, label='Non-Toxic preds')
ax.hist(probs[probs >= 0.5], bins=30, color='#ef4444', alpha=0.7, label='Toxic preds')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary')
ax.set_xlabel('Toxicity Probability')
ax.set_ylabel('Count')
ax.set_title('Test Set — Predicted Probability Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('prediction_distribution.png', bbox_inches='tight')
plt.show()

submission.head(10)

## 💾 Cell 11 — Save Model & Tokenizer

In [ ]:
SAVE_DIR = './toxguard_final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model and tokenizer saved to: {SAVE_DIR}')

## 🔍 Cell 12 — XAI: LIME Token-Level Explanations

> **Explainable AI (XAI)** — LIME perturbs the input text and fits a local linear model
> to explain *which words pushed the model toward Toxic or Non-Toxic*.
> Green bars = evidence for Non-Toxic. Red bars = evidence for Toxic.

In [ ]:
from lime.lime_text import LimeTextExplainer

# Load saved model for XAI (works even if run independently)
SAVE_DIR    = './toxguard_final'
_tokenizer  = AutoTokenizer.from_pretrained(SAVE_DIR)
_model      = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
_model.eval().to(DEVICE)

def predict_proba_lime(texts):
    """Return [P(non-toxic), P(toxic)] for a list of texts — required by LIME."""
    enc = _tokenizer(
        list(texts), return_tensors='pt',
        truncation=True, padding=True, max_length=128
    ).to(DEVICE)
    with torch.no_grad():
        logits = _model(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

explainer = LimeTextExplainer(class_names=['Non-Toxic', 'Toxic'])

LIME_EXAMPLES = [
    ('Toxic',     "I hope she gets what she deserves, stupid bitch."),
    ('Non-Toxic', "She is one of the best actresses in Bollywood!"),
    ('Toxic',     "California would be a better place without all the dirty mexicans."),
    ('Non-Toxic', "यह एक अच्छा काम है, बधाई हो!"),
]

fig, axes = plt.subplots(len(LIME_EXAMPLES), 1,
                          figsize=(12, 3.5 * len(LIME_EXAMPLES)))
fig.suptitle('XAI — LIME Token Importance', fontsize=14, fontweight='bold')

for i, (true_label, text) in enumerate(LIME_EXAMPLES):
    exp   = explainer.explain_instance(text, predict_proba_lime,
                                       num_features=10, num_samples=300,
                                       labels=[1])   # explain Toxic class
    words = exp.as_list(label=1)
    tokens, scores = zip(*words) if words else ([], [])
    bar_colors = ['#ef4444' if s > 0 else '#22c55e' for s in scores]

    ax = axes[i] if len(LIME_EXAMPLES) > 1 else axes
    ax.barh(range(len(tokens)), scores,
            color=bar_colors, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    prob = predict_proba_lime([text])[0][1]
    ax.set_title(f'[{true_label}] "{text[:70]}..."  →  P(toxic)={prob:.2%}',
                 fontsize=9)
    ax.set_xlabel('LIME importance (→ Toxic)')
    toxic_patch    = mpatches.Patch(color='#ef4444', label='Pushes → Toxic')
    nontoxic_patch = mpatches.Patch(color='#22c55e', label='Pushes → Non-Toxic')
    ax.legend(handles=[toxic_patch, nontoxic_patch], loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('xai_lime.png', bbox_inches='tight')
plt.show()
print('✅ Saved xai_lime.png')

## 🔥 Cell 13 — XAI: Attention Heatmap

> Visualises the **last-layer, first-head attention weights** from XLM-RoBERTa.
> Darker cells = the model paid more attention to that token pair when classifying.

In [ ]:
ATTN_EXAMPLES = [
    "I hope she gets what she deserves, stupid bitch.",
    "She is one of the best actresses in Bollywood!",
]

# Reload with output_attentions=True
_model_attn = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR, output_attentions=True
).eval().to(DEVICE)

fig, axes = plt.subplots(1, len(ATTN_EXAMPLES),
                          figsize=(7 * len(ATTN_EXAMPLES), 6))
fig.suptitle('XAI — Attention Heatmap (last layer, head 0)', fontsize=13, fontweight='bold')

for i, text in enumerate(ATTN_EXAMPLES):
    inputs = _tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=32, padding=False
    ).to(DEVICE)
    with torch.no_grad():
        outputs = _model_attn(**inputs)
    tokens_ids = inputs['input_ids'][0]
    tok_labels = _tokenizer.convert_ids_to_tokens(tokens_ids)
    tok_labels = [t.replace('▁', '') for t in tok_labels]

    # last layer, head 0 attention
    attn = outputs.attentions[-1][0, 0].cpu().numpy()  # shape (seq, seq)
    prob = torch.softmax(outputs.logits, dim=-1)[0][1].item()

    ax = axes[i] if len(ATTN_EXAMPLES) > 1 else axes
    sns.heatmap(attn, xticklabels=tok_labels, yticklabels=tok_labels,
                cmap='Reds', ax=ax, linewidths=0.3, linecolor='white',
                cbar_kws={'shrink': 0.7})
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    ax.set_title(f'P(toxic)={prob:.2%}\n"{text[:50]}"', fontsize=9)

plt.tight_layout()
plt.savefig('xai_attention.png', bbox_inches='tight')
plt.show()
print('✅ Saved xai_attention.png')

## 🎨 Cell 14 — Gradio Demo (with XAI)

> Run this cell after training. It launches an interactive web app where you can
> type any text in any language and get an instant toxicity prediction **plus LIME explanation**.
> `share=True` gives a public URL valid for 1 week.

In [ ]:
import gradio as gr
from lime.lime_text import LimeTextExplainer as _LimeExp
import matplotlib
matplotlib.use('Agg')
import io, base64

SAVE_DIR   = './toxguard_final'
_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
_model_gr  = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).eval().to(DEVICE)

LABELS = [
    (0.0,  0.3,  '✅ Non-Toxic',  '#22c55e'),
    (0.3,  0.6,  '⚠️ Borderline', '#f59e0b'),
    (0.6,  1.01, '🚨 Toxic',       '#ef4444'),
]

_lime_exp = _LimeExp(class_names=['Non-Toxic', 'Toxic'])

def _predict_proba(texts):
    enc = _tokenizer(list(texts), return_tensors='pt',
                     truncation=True, padding=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        logits = _model_gr(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

def _lime_chart(text, prob):
    """Return a base64 PNG of the LIME bar chart."""
    exp   = _lime_exp.explain_instance(text, _predict_proba,
                                       num_features=8, num_samples=200, labels=[1])
    words = exp.as_list(label=1)
    if not words:
        return ''
    tokens, scores = zip(*words)
    colors = ['#ef4444' if s > 0 else '#22c55e' for s in scores]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(range(len(tokens)), scores, color=colors, edgecolor='white')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — Token Importance  (P(toxic)={prob:.2%})', fontsize=10)
    ax.set_xlabel('← Non-Toxic  |  Toxic →')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    return f'<img src="data:image/png;base64,{b64}" style="width:100%;border-radius:8px;"/>'

def classify_text(text: str):
    if not text.strip():
        return '<p>Please enter some text.</p>', ''
    prob = _predict_proba([text])[0][1]
    verdict, hex_col = '✅ Non-Toxic', '#22c55e'
    for lo, hi, label, color in LABELS:
        if lo <= prob < hi:
            verdict, hex_col = label, color
            break
    badge = f"""
    <div style='font-family:sans-serif;padding:16px;border-radius:10px;
                background:#1e1e2e;color:#cdd6f4;'>
      <h2 style='margin:0 0 8px;color:{hex_col};'>{verdict}</h2>
      <p style='margin:0;font-size:18px;'>
        Toxicity Probability: <strong style='color:{hex_col};'>{prob:.2%}</strong>
      </p>
      <div style='margin-top:12px;background:#313244;border-radius:6px;height:16px;width:100%;'>
        <div style='height:16px;border-radius:6px;width:{prob*100:.1f}%;background:{hex_col};'></div>
      </div>
    </div>
    """
    lime_html = _lime_chart(text, prob)
    return badge, lime_html

with gr.Blocks(theme=gr.themes.Soft(), title='🛡️ ToxGuard') as demo:
    gr.Markdown('# 🛡️ ToxGuard — Multilingual Toxicity Detector')
    gr.Markdown('Powered by **XLM-RoBERTa** + **LIME XAI**. Best Val ROC-AUC: **0.9918**')
    with gr.Row():
        txt_input = gr.Textbox(
            label='Enter Text (any language)',
            placeholder='Type a comment in English, Hindi, Spanish, etc...',
            lines=4
        )
    btn = gr.Button('🔍 Analyse', variant='primary')
    with gr.Row():
        verdict_out = gr.HTML(label='Verdict')
        lime_out    = gr.HTML(label='XAI — LIME Explanation')
    btn.click(classify_text, inputs=txt_input, outputs=[verdict_out, lime_out])
    gr.Examples(
        examples=[
            ['She is one of the best actresses in Bollywood!'],
            ['I hope she gets what she deserves, stupid bitch.'],
            ['यह एक अच्छा काम है, बधाई हो!'],
            ['California would be a better place without all the dirty mexicans'],
            ['The weather today is really beautiful and sunny.'],
        ],
        inputs=txt_input
    )

demo.launch(share=True)  # share=True → public URL valid for 1 week